# Chapter 7: Claim Verification and Calibrated Abstention

A report that cites a number should cite the right number for the right object, and a verifier checks this by comparing the claim with reference evidence. Chapter 7 develops that comparison: binding a claim to a scoped reference, comparing typed values under a declared rule and keeping contradiction, missing evidence and malformed input distinct. This notebook applies the rule to claims about the mixture-model comparison and the regional medoid search, using the evidence store built in Chapter 6.

The notebook runs a regression suite of authored claims, replays the chapter's mixture-model mappings from their source records, distinguishes wrong values from unmeasured results, damages copies of a record to test integrity handling and examines what a Supported verdict does and does not establish. No model is fitted and no language model is called.

**Evidence.** The store is `book/evidence/parallel_aml_20260921/store.json`. The mixture-model mappings and saved verdicts are in `book/evidence/aml_current/`. Both studies use synthetic AML populations.

**Run order.** Run the cells from top to bottom. Each assertion checks a recomputed verdict against its expected or saved value.

## Setup and source checks

Use a Python kernel with the dependencies listed in the README. The setup cell locates the data root, which holds the evidence files and helper modules. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package.

`load()` checks the digests that link the regional results, frozen finalists, protocol and admission records. The verifier `verify`, the regression cases `cases`, the pointer resolver `resolve` and the comparison `equal` are imported from `book/parallel_evidence/build.py` under the data root.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT))
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Exercise the claim contract

A verifier compares a claim with reference evidence and returns a verdict. Chapter 7 separates four outcomes (Section 7.2). A claim is Supported when a reference exists at its address and the values agree under the declared comparison. It is Contradicted when the reference exists and the values differ. It is Absent when the store holds no reference at the address, which describes the available evidence and does not show the claim to be false. It is Unscorable when the claim is malformed, for example when the value has the wrong type or shape, so that no evidential comparison can be made.

`verify(store, claim)` implements this contract for claims supplied as an explicit identifier and a typed value. It first checks the source file's digest and pointer, as in Chapter 6. For a continuous value with reference $x(a)$ and claimed value $\hat x$ it declares agreement when $|\hat x-x(a)|\le\tfrac12\times10^{-6}$, which accepts six-decimal display rounding and nothing more; counts must match exactly, and lists must agree in length and element by element.

`cases()` supplies a regression suite of 17 authored claims with expected verdicts. They cover correct values for both studies, a wrong value, a correct number attached to the wrong metric, split or study, requests for unmeasured results and three malformed claims. The cell runs each claim, asserts that the verdict equals its expected answer and counts the verdicts.

In [2]:
from book.parallel_evidence.build import OUT, cases, verify
store=read(OUT/"store.json")
rows=[]
for name,id_,value,expected in cases():
    verdict=verify(store,{"id":id_,"value":value})["verdict"]
    assert verdict==expected
    rows.append(dict(case=name,id=id_,value=value,expected=expected,actual=verdict))
suite=pd.DataFrame(rows)
display(suite)
counts=suite["actual"].value_counts().reindex(["Supported","Contradicted","Absent","Unscorable"])
assert counts.tolist()==[7,4,3,3]
display(counts.rename("cases"))

,case,id,value,expected,actual
0,moe_correct,moe:confirmation/llm_compromise/worst_logloss,0.103333,Supported,Supported
1,moe_false,moe:confirmation/llm_compromise/worst_logloss,0.400000,Contradicted,Contradicted
2,moe_missing,moe:experiment/fairness_passed,True,Absent,Absent
3,regional_loss,region:confirmation/reflection_73101/brier,0.072851,Supported,Supported
4,regional_contrast,region:confirmation/reflection_73101/contrast,0.058326,Supported,Supported
5,regional_share,region:confirmation/reflection_73101/share,0.115220,Supported,Supported
6,regional_count,region:protocol/confirmation_n,200000,Supported,Supported
7,admission,region:admission/admitted,True,Supported,Supported
8,paired_interval,region:comparison/reflection-continuation_7310...,"[-0.001723, 0.002495]",Supported,Supported
9,wrong_metric,region:confirmation/reflection_73101/brier,0.058326,Contradicted,Contradicted


actual
Supported       7
Contradicted    4
Absent          3
Unscorable      3
Name: cases, dtype: int64

**Read the output.** All 17 verdicts match their expected answers: seven Supported, four Contradicted, three Absent and three Unscorable. The three Contradicted rows `wrong_metric`, `wrong_split` and `wrong_study` all claim 0.058326, the correct confirmation contrast of the seed-73101 reflection region, at addresses holding the regional Brier loss, the discovery contrast and a mixture-model logloss. A number that is correct somewhere is therefore not accepted at the wrong address. The Unscorable rows show that the string `"0.058326"`, the boolean `True` as a sample size and a one-element interval are rejected before comparison. The expected verdicts are authored answers for this suite, so the result is a contract test of the implementation, not an estimate of verification accuracy on free text.

## Replay the mixture-model mappings

The chapter's worked example applies the same contract to the mixture-model comparison of Appendix B. It supplies six explicit mappings of entity, field and claimed value against the current projection in `book/evidence/aml_current/`: four that match their records, one false loss and one request for a fairness assessment that the study did not perform. The saved file `claim_verdicts.json` holds each mapping with its verdict and evidence record.

The cell does not trust the saved verdicts. For each mapping with evidence it rechecks the source digest, resolves the pointer, confirms that the recorded value is unchanged and recomputes the verdict with the same comparison `equal`. A mapping without evidence receives Absent. The final assertion requires the recomputed lines to equal the saved output that the chapter prints.

In [3]:
from book.parallel_evidence.build import resolve, equal
from companion import sha
saved=read(ROOT/"book/evidence/aml_current/claim_verdicts.json")
lines=[]
for row in saved:
    evidence=row["evidence"]
    if evidence is None:
        verdict="Absent"
    else:
        assert sha(ROOT/evidence["source"])==evidence["source_sha256"]
        value=resolve(read(ROOT/evidence["source"]),evidence["pointer"])
        assert value==evidence["value"]
        verdict="Supported" if equal(row["claimed"],value) else "Contradicted"
    assert verdict==row["verdict"]
    lines.append(f"{verdict}: {row['entity']} {row['field']} {row['claimed']}")
assert "\n".join(lines)==(ROOT/"book/evidence/aml_current/claim_output.txt").read_text().strip()
print("\n".join(lines))

Supported: llm_compromise overall_logloss 0.09826
Supported: llm_compromise worst_logloss 0.103333
Supported: experiment confirmation_n 200000
Supported: full_compromise worst_logloss 0.103417
Contradicted: llm_compromise worst_logloss 0.4
Absent: experiment fairness_passed True


**Read the output.** The six recomputed lines reproduce the chapter listing. The claimed value 0.09826 is Supported for the overall logloss of `llm_compromise` because the recorded value 0.098260 agrees within the display tolerance. The claim of 0.4 for that endpoint's worst-population logloss is Contradicted by the record 0.103333. The fairness request is Absent: the store documents that no fairness assessment was recorded, which is a gap in the evidence and not a finding that the model is unfair or fair.

## Distinguish wrong from missing

A Contradicted verdict and an Absent verdict call for different actions. A contradiction means that the claim misstates a recorded measurement and should be corrected. An absence means that the claim asserts something the study did not measure, so either the claim is removed or new evidence is required. The cell supplies the regional contrast 0.058326 at its own address and at the Brier-loss address of the same region, and then requests two results the experiment did not measure: a predictor-repair gain and a confirmed fragmentation mechanism, the explanation behind the Chapter 5 proposal.

In [4]:
claim={"id":"region:confirmation/reflection_73101/contrast","value":0.058326}
print(verify(store,claim)["verdict"])
claim["id"]="region:confirmation/reflection_73101/brier"
print(verify(store,claim)["verdict"])
for id_,value in [("region:experiment/predictor_repair_gain",.01),("region:experiment/fragmentation_cause_confirmed",True)]:
    print(id_,verify(store,{"id":id_,"value":value})["verdict"])

Supported
Contradicted
region:experiment/predictor_repair_gain Absent
region:experiment/fragmentation_cause_confirmed Absent


**Read the output.** The value 0.058326 is Supported as the contrast and Contradicted as the Brier loss, whose record is 0.072851; a reference exists at both addresses. The repair gain and the fragmentation mechanism have no records, so both requests are Absent. The medoid search measured where the frozen predictor has elevated loss; it neither retrained the predictor nor tested why the region's errors arise. A report that claimed either result would be making an unsupported claim, and the verifier identifies it as such without calling it false.

## Test evidence integrity

A verdict is meaningful only if the reference itself is intact. If a stored record no longer matches its source, returning Absent or Contradicted would misreport a damaged pipeline as a property of the claim. The verifier therefore raises an error instead of returning a verdict. The cell copies the store and damages one record in two ways: the stored source digest is replaced with `"damaged"`, and the stored value is replaced with −1. It asserts that verification of an otherwise correct claim raises `ValueError` in both cases. The saved store is not modified.

In [5]:
from copy import deepcopy
key="region:confirmation/reflection_73101/contrast"
for field,bad in [("source_sha256","damaged"),("value",-1.)]:
    broken=deepcopy(store);broken["facts"][key][field]=bad
    try:
        verify(broken,{"id":key,"value":.058326})
    except ValueError as e:
        print(type(e).__name__,str(e))
    else:
        raise AssertionError("Corrupt record accepted")

ValueError Source hash mismatch: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json
ValueError Source pointer/value mismatch: region:confirmation/reflection_73101/contrast


**Read the output.** The damaged digest produces a source hash mismatch, and the altered value produces a pointer/value mismatch, because the source file still holds the original contrast. In both cases verification stops. A corrupted record is a pipeline failure to be repaired before any claim is assessed, so the verifier issues no Absent or Contradicted verdict for it.

## Interpret support

A Supported verdict establishes that a claim transcribes its record correctly. What the record means is a separate question. The cell verifies two records that concern the Chapter 5 proposal: that the union rule was admitted, and the paired reflection-minus-continuation interval at seed 73101. It then reads the seed-73102 confirmation contrasts of both branches, which bear on a claim that the structural branch produced the larger contrast.

In [6]:
for id_ in ["region:admission/admitted","region:comparison/reflection-continuation_73101/simultaneous_ci"]:
    value=store["facts"][id_]["value"]
    print(id_,value,verify(store,{"id":id_,"value":value})["verdict"])
display(pd.Series({arm:store["facts"][f"region:confirmation/{arm}_73102/contrast"]["value"] for arm in ["reflection","continuation"]},name="confirmation contrast, seed 73102"))

region:admission/admitted True Supported
region:comparison/reflection-continuation_73101/simultaneous_ci [-0.0017229699434947615, 0.002494938298177806] Supported


reflection     0.053993
continuation   0.057581
Name: confirmation contrast, seed 73102, dtype: float64

**Read the output.** Both records are Supported. Admission establishes that the rule could be executed and scored, and the correctly transcribed interval, [−0.001723, 0.002495], contains zero, so neither record supports a claim that the admitted rule improved on continuation. At seed 73102 the structural branch has contrast 0.053993 and continuation 0.057581. A report sentence claiming that the structural branch had the larger point estimate at every seed would therefore be contradicted by these records, even though each individual number it cited could be Supported. Checking numbers against records is necessary for a faithful report but does not by itself check the report's comparative statements.

## What the replay establishes

The verifier has been exercised on 17 authored claims, six mixture-model mappings, four targeted regional claims and two corruption checks. It distinguishes a correct value, a value at the wrong address, an unmeasured result and a malformed claim. It stops when a reference is damaged.

Every claim here was supplied with an explicit identifier and typed value. Extracting such claims from free text, binding a phrase such as "the weak region's loss" to the right address and judging whether a sentence interprets a correct number correctly are separate steps that these checks do not measure. Chapter 8 tests verification under adversarially varied inputs, and Chapter 10 applies it to generated report text.


## What to try

1. **Typed values.** Before running it, predict the verdict for `{"id": "region:confirmation/reflection_73101/n", "value": 23044.0}`. Then run it and explain the result from the comparison rule for counts.
2. **Tolerance.** Find the smallest change to 0.058326 that turns the contrast claim from Supported to Contradicted, and explain why the tolerance reflects display rounding rather than sampling uncertainty.
3. **Wrong and missing.** Write one false claim and one unmeasured claim about the seed-73103 union-branch region. Verify both and state what a report author should do in each case.
4. **Scope.** Explain why a Supported verdict for every number in a paragraph does not establish that the paragraph is faithful to the evidence.
